In [ ]:
# Databricks notebook source
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

# Shared price-message contract for AEMO, ENTSO-E, and Elecz.
schema = StructType() \
    .add("timestamp", StringType()) \
    .add("zone", StringType()) \
    .add("price", DoubleType()) \
    .add("source", StringType())

aiven_password = dbutils.secrets.get(scope="kafka-scope", key="aiven-password")
jaas_config = f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username="avnadmin" password="{aiven_password}";'

kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka-26289383-md-f1a0.l.aivencloud.com:26452")
    .option("subscribe", "energy-prices")
    .option("startingOffsets", "latest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("kafka.ssl.truststore.location", "/Volumes/workspace/default/energy_realtime/aiven_ca.pem")
    .option("kafka.ssl.truststore.type", "PEM")
    .load()
)

parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

query = (parsed_df.select("timestamp", "zone", "price", "source").writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/workspace/default/energy_realtime/checkpoints")
    .toTable("energy_prices_realtime"))

query.awaitTermination()